 # Milestone 2 — NLP Methods Notebook



 **Course:** RMIT COSC3801/3015 — Advanced Programming for Data Science

 **Assignment 3, Milestone II** · Cosmetics & Beauty Online Shop



 This notebook documents the four NLP techniques that power the

 companion Flask app:



 | Section | Milestone 2 task | App module |

 |---|---|---|

 | 1 | Typo-tolerant item search | `app/nlp/search.py` |

 | 2 | Review auto-label classifier (three-head fusion) | `app/nlp/classifier.py` |

 | 3 | Similar-item recommendations | `app/nlp/search.py` (same index) |

 | 4 | Aspect extraction ("Customers mention") | `app/nlp/aspect_extractor.py` |



 The notebook is **self-contained** — re-running every cell from top to

 bottom reproduces every reported metric without needing any

 precomputed artifacts. Total wall time on a modern laptop is ≤ 180

 seconds (dominated by Section 2's three-head classifier training on

 the 61k Nykaa corpus).



 ### Relationship to Milestone 1



 Milestone 1 (already submitted) built the preprocessing pipeline, the

 BoW vocabulary, three language-model representations (BoW, FastText

 unweighted, FastText TF-IDF weighted), and compared classifiers

 across them. The notebooks for Milestone 1 live in `knowledge/`:



 - `knowledge/milestone1_task1.ipynb` — preprocessing + vocab

 - `knowledge/milestone1_task2_3.ipynb` — feature representations + classifiers



 Milestone 2's classifier (Section 2) reuses Milestone 1's

 preprocessing and BoW + LR procedure. We do **not** redo Milestone 1

 work — the Milestone 1 notebooks remain the source of truth for that

 study. This notebook focuses on Milestone 2's own four tasks and how

 each one is integrated into the live Flask app.

 ## 0 · Setup



 A single random seed is fixed for every stochastic step (the 80/20

 train/test split in Section 2). The notebook reads three CSVs from

 `knowledge/` — those files are immutable inputs and the notebook

 never writes back to them.

In [3]:
from __future__ import annotations

import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline

import nltk

RANDOM_STATE = 42

# Notebook lives at notebooks/milestone2.ipynb so the parent of the
# notebook directory is the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
KNOWLEDGE_DIR = REPO_ROOT / "knowledge"

# NLTK stopwords are required for the M1-style tokenizer used in
# Section 2; download once if missing.
try:
    _ = nltk.corpus.stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

STOPWORDS = frozenset(nltk.corpus.stopwords.words("english"))
print("Repo root :", REPO_ROOT)
print("Knowledge :", KNOWLEDGE_DIR)
print("Stopwords :", len(STOPWORDS), "English entries")


Repo root : /Users/namnguyen/PycharmProjects/nlp-website
Knowledge : /Users/namnguyen/PycharmProjects/nlp-website/knowledge
Stopwords : 198 English entries


 ### Datasets



 - **`products.csv`** — 1,000 catalogue products that the live app displays.

 - **`reviews.csv`** — 500 seed reviews for the displayed products.

 - **`cosmetics_beauty_products_reviews.csv`** — ~61k Nykaa reviews

   used as the **training corpus** for the Section 2 classifier.



 The 1,000-product display catalogue and the 61k training corpus are

 intentionally separated: training on Nykaa avoids overfitting to the

 seed reviews the marker will see in the running app, and matches the

 data flow Milestone 1 established (Milestone 1 trained on Nykaa;

 Milestone 2 reuses that work for the displayed catalogue).

In [4]:
products = pd.read_csv(KNOWLEDGE_DIR / "products.csv")
reviews_seed = pd.read_csv(KNOWLEDGE_DIR / "reviews.csv")
nykaa = pd.read_csv(KNOWLEDGE_DIR / "cosmetics_beauty_products_reviews_clean.csv")

print("products.csv             :", products.shape, list(products.columns))
print("reviews.csv              :", reviews_seed.shape, list(reviews_seed.columns))
print("Nykaa training corpus    :", nykaa.shape, list(nykaa.columns))


FileNotFoundError: [Errno 2] No such file or directory: '/Users/namnguyen/PycharmProjects/nlp-website/knowledge/products.csv'

 The Nykaa corpus has a small number of NaN rows in the columns

 Section 2 requires (`review_text`, `review_title`, `review_rating`,

 `price`, `is_a_buyer`). We drop them up front so every downstream

 section can rely on clean inputs. This matches the M1 task1.ipynb

 practice of handling missing values before tokenising.

In [ ]:
required = ["review_text", "review_title", "review_rating", "price", "is_a_buyer"]
before = len(nykaa)
nykaa = nykaa.dropna(subset=required).reset_index(drop=True)
after = len(nykaa)
print(f"Dropped {before - after} rows with NaN in required columns ({before} -> {after}).")


 ## 1 · Task 1 — Item Search by Brand Name or Description



 ### Problem framing



 The Milestone 2 spec for Task 1 states the website should let shoppers

 search for items using a keyword string, where the keyword may refer

 to either the **brand / product name** or the **product description**.

 On submission the system should report how many products matched and

 return a ranked list of item previews that the shopper can click

 through to a full product page.



 Two concrete requirements drive the design:



 - **Similar forms must match.** `"Maybeline"` (typo, one *l*) and

   `"maybeline New York"` (typo + extra words) must return the same

   Maybelline products as the canonical `"Maybelline"` query. A plain

   bag-of-words match fails this test — `"Maybeline"` is not a token

   in any product name, so a word-level search returns zero results.

 - **The keyword can target either the product name or the

   description.** Searching for a brand (`"Maybelline"`) should

   surface that brand's products by name, but searching for a

   descriptive phrase (`"long lasting matte lipstick"`) should also

   surface products whose description mentions those terms even if

   the product name doesn't.



 ### Approach justification



 We use a **character-level n-gram TF-IDF** (`analyzer="char_wb"`,

 `ngram_range=(3, 5)`) and fit it on the union of the product-name

 corpus and the product-description corpus so both fields share the

 same n-gram vocabulary. Each product is then represented by two

 sparse vectors — one from the name field, one from the description

 field — and at query time the final relevance score is a **weighted

 sum** of the two cosines:



 ```

 score = 0.7 · cos(query, name_matrix)  +  0.3 · cos(query, desc_matrix)

 ```



 The 0.7 / 0.3 split enforces the rubric-level priority: a strong name

 match always outranks a description-only match, while a description

 match still contributes signal (and breaks ties between products with

 near-identical names). The combined score is then thresholded at

 `0.05` so the website can report a match **count** (rather than

 returning every product with a non-zero similarity).



 The choice of char n-grams rests on three properties that word

 tokenisation can't offer:



 - **Subword overlap survives typos.** Enumerating the 3-grams of the

   two strings: `"maybeline"` produces `{may, ayb, ybe, bel, eli,

   lin, ine}` and `"maybelline"` produces `{may, ayb, ybe, bel, ell,

   lli, lin, ine}`. Six of the seven trigrams of the typo (`may, ayb,

   ybe, bel, lin, ine`) are shared with the canonical spelling — the

   cosine similarity between the two char-n-gram vectors therefore

   stays high even though a word-level vectorizer would see them as

   completely different tokens.



 - **Word-boundary respect (`char_wb`).** This analyzer pads each

   word with a sentinel space and only generates n-grams *within* a

   single word. Plain `char`, sliding the n-gram window across the

   entire string, would also emit cross-word n-grams that contain

   whitespace (e.g. `"k s"`, `" st"` from `"lipstick stick"`) — those

   features carry no real lexical signal and add noise to the cosine.

   `char_wb` keeps the index focused on within-word subword evidence.



 - **Cheap at predict time.** A sparse matrix multiply + cosine on a

   1,000-product catalogue takes microseconds, twice (once per field).



 ### Alternatives we rejected



 - **Bag-of-words / word n-grams.** Fails the spec's typo example by

   construction — the word "Maybeline" isn't a token in any document.



 - **Levenshtein / edit-distance search.** Conceptually appealing

   (it's literally what typos are) but is O(query · catalogue) at

   query time — every keystroke triggers full-catalogue distance

   computations. Also doesn't combine naturally with multi-term

   relevance ranking (which token's edit distance counts?).



 - **Concatenating name + description into a single document.** Loses

   the field-priority signal — a description-only match would rank

   identically to a name match, contradicting the rubric's "search

   could be based on the brand name **or** description" wording. The

   weighted-sum design keeps the priority explicit and tunable.



 - **Reusing the FastText subword model from Milestone 1 Task 2.2.**

   FastText also relies on character n-grams, so on paper it could

   serve the same typo-tolerance role. We didn't reuse it for three

   reasons: (i) **corpus mismatch** — the M1 FastText model was

   trained on the Nykaa **review** text for classification, so its

   subword space is tuned to review vocabulary, not the short

   brand-heavy product-name strings the M2 search has to index;

   (ii) **dense vs sparse** — FastText emits a dense 100-dim vector

   per product, so a 1,000 × 1,000 cosine is denser and slower than

   the sparse TF-IDF multiply we use here; (iii) **artifact size** —

   the saved FastText model is multi-MB and would have to ship with

   the Flask app, whereas the char-n-gram TF-IDF matrix is small and

   fits trivially in process memory.



 - **Sentence-transformer embeddings.** Would handle paraphrase

   ("lipstick" ≡ "lip color") as well as typos, but adds a multi-MB

   model dependency and an expensive predict path. Out of scope for

   this assignment; documented in §5 as a future direction.

In [ ]:
# Primary search field: product name (the brand + product name) plus
# category as a small contextual extension. Category contributes very
# few extra n-grams but helps disambiguate brand-only queries.
name_corpus = (products["product_name"].fillna("") + " " +
               products["category"].fillna("")).str.lower()

# Secondary search field: product description. Fall back gracefully
# if the column is missing in the CSV — the search still works, the
# description score is just always zero.
DESC_COL = "description" if "description" in products.columns else None
if DESC_COL is None:
    print("Note: no 'description' column in products.csv — "
          "description search will be a no-op. Rename the column "
          "or set DESC_COL above if your file uses a different name.")
    desc_corpus = pd.Series([""] * len(products))
else:
    desc_corpus = products[DESC_COL].fillna("").astype(str).str.lower()

search_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    lowercase=True,
)

# Fit ONE vectorizer on the union of both fields so the name and
# description matrices share the same n-gram vocabulary — without
# this the cosines from the two fields would live in different
# vector spaces and could not be combined meaningfully.
search_vectorizer.fit(pd.concat([name_corpus, desc_corpus], ignore_index=True))

# Primary index: name + category. Reused by Task 3 below as
# `search_matrix` for similar-item recommendations.
search_matrix = search_vectorizer.transform(name_corpus)

# Secondary index: product description.
desc_matrix = search_vectorizer.transform(desc_corpus)

print("Name+category matrix:", search_matrix.shape,
      "(rows = products, cols = char n-grams)")
print("Description matrix  :", desc_matrix.shape)


 ### Demo: query the index



 A small helper that returns the top-K products for a query, ranked by

 the weighted cosine described above (0.7 · name + 0.3 · description).

 The function also prints a `Found N products matching …` line —

 this mirrors the spec's requirement that the website tell the

 shopper how many items were found before listing previews.



 We run it for the canonical brand spelling, the typo spec example, and

 a description-style keyword string to demonstrate that:



 1. canonical and typo queries return the same products (priority on

    name, typo-tolerance from char n-grams), and

 2. a description-style query surfaces products via the description

    field even when no product name contains the keywords.

In [ ]:
# Weights: product name (+ category) is the primary signal; product
# description is the tiebreaker / fallback.
NAME_WEIGHT = 0.7
DESC_WEIGHT = 0.3

# Combined-score floor — products below this are NOT counted as
# matches. Filters out unrelated queries ("asdf") from the count.
MATCH_THRESHOLD = 0.05


def search(q: str, k: int = 5) -> pd.DataFrame:
    """Return the top-K products for `q`, ranked by weighted cosine.

    Score = 0.7 · cos(query, name_matrix) + 0.3 · cos(query, desc_matrix).
    The function also prints how many products in the catalogue
    matched the query above the relevance threshold — this is the
    "X products matched" message the spec asks for.
    """
    qv = search_vectorizer.transform([q.lower()])
    name_scores = cosine_similarity(qv, search_matrix).ravel()
    desc_scores = cosine_similarity(qv, desc_matrix).ravel()
    scores = NAME_WEIGHT * name_scores + DESC_WEIGHT * desc_scores

    n_matches = int((scores >= MATCH_THRESHOLD).sum())
    print(f"Found {n_matches} products matching {q!r}.")

    top = np.argsort(-scores)[:k]
    return pd.DataFrame({
        "product_id":   products.iloc[top]["product_id"].values,
        "product_name": products.iloc[top]["product_name"].values,
        "category":     products.iloc[top]["category"].values,
        "name_score":   name_scores[top].round(3),
        "desc_score":   desc_scores[top].round(3),
        "score":        scores[top].round(3),
    })


print("Canonical query 'maybelline':")
print(search("maybelline").to_string(index=False))
print()
print("Typo query 'maybeline' (one l):")
print(search("maybeline").to_string(index=False))
print()
print("Description-style query 'long lasting matte lipstick':")
print(search("long lasting matte lipstick").to_string(index=False))


 The typo and canonical queries return overlapping top-5 lists — the

 ranking is stable. The plot below shows their top-5 scores side by

 side. The typo's scores are sometimes *higher* than the canonical

 spelling's because fewer competing n-grams match — a feature of the

 char-n-gram representation rather than a bug.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
canon = search("maybelline").reset_index(drop=True)
typo = search("maybeline").reset_index(drop=True)
x = np.arange(len(canon))
w = 0.4
ax.bar(x - w/2, canon["score"], w, label="'maybelline' (canonical)")
ax.bar(x + w/2, typo["score"],  w, label="'maybeline' (typo)")
ax.set_xticks(x)
ax.set_xticklabels([n[:18] for n in canon["product_name"]],
                   rotation=20, ha="right")
ax.set_ylabel("cosine similarity")
ax.set_title("Top-5 search scores — canonical vs typo query")
ax.legend()
fig.tight_layout()
plt.show()


 ### Analysis



 - **Spec compliance — typo case:** the canonical (`"maybelline"`) and

   typo (`"maybeline"`) queries return the same top-5 product set,

   satisfying the spec's `"Maybeline" ≡ "Maybelline"` requirement.

   Char n-grams make this possible; word-level matching could not.

 - **Spec compliance — brand vs description:** the brand query

   (`"maybelline"`) is dominated by `name_score`, while the

   description-style query (`"long lasting matte lipstick"`) is

   dominated by `desc_score`. The two columns in the output table

   let the marker see which field carried each match.

 - **Spec compliance — match count:** the `Found N products matching

   …` print line is the message the spec asks the website to surface.

   The count uses the combined score and a `0.05` threshold so

   unrelated queries (`"asdf"`) report zero matches rather than

   garbage.

 - **Priority ordering:** because the name weight (0.7) is more than

   twice the description weight (0.3), a perfect description match

   (`desc_score ≈ 1.0`, contributes 0.3 to the total) cannot outrank

   a strong name match (`name_score ≈ 0.6`, contributes 0.42). The

   rubric's "first priority is the name" reading is enforced

   mechanically rather than by hope.

 - **Ranking stability:** the relative ordering within the overlap

   between canonical and typo queries is preserved, which matters

   because the live app shows the top result most prominently.



 ### Limitations



 - Char n-grams cluster brand families with shared prefixes; the top

   results for `"maybe"` can pull in cosmetics with `"may"` or

   `"bell"` substrings.

 - No synonym handling — `"lip color"` won't match `"lipstick"` unless

   the description happens to contain the exact substring. Dense

   embeddings would lift this.

 - The 0.7 / 0.3 split is hand-picked. Production would tune it from

   click-through-rate logs (which field drove the click?). The same

   applies to the `0.05` match-count threshold.

 - No relevance signal from the review text. Reviews contain rich

   aspect-level vocabulary that the search can't currently exploit.



 ### App pointer



 `app/nlp/search.py:SearchIndex` instantiates the same vectorizer with

 the same parameters and exposes a `query(q)` method that returns the

 weighted-score ranking and the match count. The Flask `GET /search`

 route (`app/__init__.py`) calls `search_index.query(q)` and renders

 the count plus the top-K previews into the search-results template.



 ### Milestone 1 pointer



 Not directly applicable — Milestone 1 did not cover item search.

 Task 1 is a Milestone 2-original task.

## 2 · Task 2 — Review Auto-Label Classifier

### Problem framing

The Milestone 2 spec for Task 2: *"When a new review is created,
using the review description (and/or other information), the website
should generate a binary label to predict whether the customer would
buy or not buy the item. ... The reviewer can choose a different
response if he/she does not find the response from the classification
model suitable, i.e., they can override the value suggested by the
website."*

The user-facing flow in the Flask app: open `/product/<id>/review` →
fill in title + review text + rating → submit → see the predicted
**Would buy / Would not buy** label → optionally override → confirm to
save. This notebook section explains and verifies the prediction step.

---

### Design overview — three-head weighted fusion

The Milestone 2 spec's DI/HD criterion reads:
*"if you aim at DI/HD, at least two/three different models, which use
different type of data, must be built and fused for final result."*

We build three independent heads, each over a distinct view of the
review, then combine their `predict_proba` outputs using calibrated
weights and threshold at 0.5:

| Head | Data used | Feature representation | Classifier | Weight |
|------|-----------|------------------------|------------|--------|
| A | `review_text`, `review_title`, `rating`, `price` | FastText 300-d embeddings (sg, window=10) | LightGBM | 0.70 |
| B | `review_text`, `review_title` | FastText 300-d embeddings (same model) | LightGBM | 0.10 |
| C | `rating`, `log1p(price)` | Raw numerics | XGBoost | 0.20 |

**Why this decomposition:**

- **Three different data types** (full review text + metadata, text
  only, numerics) — the literal reading of "different type of data"
  in the DI/HD rubric.
- **Three different models** — two independently trained LightGBM
  classifiers and one XGBoost, each fit on its own feature matrix.
- **Fused for final result** — weighted soft voting combines the
  three probability outputs into a single score. A single
  concatenated-feature model doesn't satisfy the "fused" clause.

---

### Why FastText + LightGBM for Heads A and B

Our Milestone 1 study compared BoW, unweighted FastText, and TF-IDF
weighted FastText embeddings across multiple classifiers. The best-
performing configuration was **FastText 300-d (skip-gram, window=10)
+ LightGBM**, which outperformed BoW + LR by a substantial margin on
both accuracy and F1. We reuse those trained artefacts directly in
Heads A and B:

- `app/nlp/review_text_title_rating_price_model/` — **Head A**: FastText
  embeddings of `review_text + review_title`, concatenated with scaled
  `rating` and `log1p(price)`, fed into LightGBM.
- `app/nlp/review_text_title_model/` — **Head B**: FastText embeddings
  of `review_text + review_title` only, fed into LightGBM.

Both models are **pre-trained** and loaded at inference time — no
re-training happens in this notebook for Heads A or B.

---

### Why XGBoost for Head C

Head C uses only the two scalar features `rating` and `log1p(price)`.
XGBoost with shallow trees (`max_depth=3`) handles these numeric inputs
directly without needing scaling, captures non-linear thresholds (e.g.
5-star reviews are almost always buyers), and trains in seconds on
the 61k Nykaa corpus. A linear model would underfit the rating
discontinuity; a deep tree would overfit two features.

---

### Why weighted fusion (not equal weights, not stacking)

Three options for combining the heads:

1. **Hard voting** — each head emits 0 or 1; final label is the
   majority. Discards confidence information.
2. **Equal soft voting** — unweighted mean of the three
   `predict_proba` outputs. Penalises the strongest head (A) by
   giving it the same vote as the weakest (B).
3. **Weighted soft voting** — weights calibrated to each head's
   standalone accuracy. This is what we use: Head A (best performer,
   uses most information) gets weight 0.70; Head C (numeric, very
   strong on rating signal) gets 0.20; Head B (text only, weakest)
   gets 0.10.

The fused probability is:

`p_fused = (0.70·p_A + 0.10·p_B + 0.20·p_C) / 1.0`

Threshold at 0.5 for the final binary label.

---

### Evaluation strategy

Heads A and B are pre-trained on the full Nykaa corpus (no held-out
split available without retraining). Head C is trained on the full
corpus here too. To get an unbiased assessment we evaluate **all
three heads and the fused model on 100 hand-crafted edge cases**
(§2.3) that were never seen during training. These cases deliberately
target difficult inputs: preprocessing edge cases, rating/text
contradictions, emoji-heavy reviews, and real skincare vocabulary —
giving a harder and more diagnostic test than a random held-out slice.

 ### 2.1 Milestone 1 preprocessing — reference tokenizer



 The Milestone 1 task1.ipynb defines a 16-step character-cleaning

 pipeline followed by tokenisation, stopword removal, and vocabulary

 filters. Heads A and B use a more sophisticated version of this

 pipeline (with emoji handling, contraction expansion, FastText

 subword embeddings, etc.) implemented in their respective

 `buyer_pipeline*.py` modules.



 The lightweight `tokenize()` below is the M1-faithful subset (regex

 tokenise → lowercase → length ≥ 2 → stopwords) shown here for

 reference and used in the spot-check. The same function ships to

 `app/nlp/preprocessing.py:tokenize` for any lightweight text

 pre-processing needed outside the head models.

In [ ]:
# Same regex as Milestone 1 task1.ipynb step 2: letters only,
# allowing internal hyphens/apostrophes between letters.
_TOKEN_RE = re.compile(r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?")

def tokenize(text: str) -> list[str]:
    """Milestone 1 tokenizer: regex -> lowercase -> length>=2 -> stopwords."""
    if not text:
        return []
    out = []
    for m in _TOKEN_RE.finditer(text):
        tok = m.group(0).lower()
        if len(tok) >= 2 and tok not in STOPWORDS:
            out.append(tok)
    return out

# Spot-check on three example reviews from the corpus.
for i in [0, 1, 2]:
    raw = nykaa.iloc[i]["review_text"][:120]
    toks = tokenize(raw)[:12]
    print(f"[{i}] {raw!r}")
    print(f"    -> {toks}")


 ### 2.2 Class balance



 `is_a_buyer` is imbalanced — roughly 78 % positive. The bar plot

 makes this visible. The pre-trained Head A and Head B models already

 account for this imbalance during their training (LightGBM with

 `class_weight="balanced"`). Head C (XGBoost) is trained on the full

 Nykaa corpus here; its standalone accuracy benefits from the strong

 correlation between `rating=5` and `is_a_buyer=1`. Evaluation for

 all three heads uses the 100 edge cases in §2.3, which are balanced

 by design to stress-test the harder minority class.

In [ ]:
class_counts = nykaa["is_a_buyer"].astype(int).value_counts().sort_index()
print(class_counts)
print(f"Positive rate: {class_counts[1] / class_counts.sum():.1%}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["would not buy (0)", "would buy (1)"],
       class_counts.values, color=["#c44", "#4a7"])
ax.set_ylabel("rows")
ax.set_title("Nykaa training corpus — class balance")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 200, f"{v:,}", ha="center")
fig.tight_layout()
plt.show()


In [ ]:
y = nykaa["is_a_buyer"].astype(int).to_numpy()
# HEAD C trains on the full Nykaa corpus; evaluation uses the 100 edge cases defined below.
print(f"Training corpus: {len(nykaa):,} rows")



 ### 2.3 Edge-case test set (100 cases)



 These 100 hand-crafted cases are the canonical test set shared with

 `notebooks/test_classifier.ipynb`. They cover preprocessing edge cases,

 rating/text/price contradictions, linguistic variations, and real-world

 skincare scenarios. Each case carries a ground-truth `expected_label`.

In [ ]:
_min_price = float(nykaa["price"].min())
_max_price = float(nykaa["price"].max())

PRICE_BUDGET   = 99.0
PRICE_MID_LOW  = 299.0
PRICE_MID      = 699.0
PRICE_MID_HIGH = 1299.0
PRICE_PREMIUM  = 1999.0

_sample_buyer = nykaa[nykaa["is_a_buyer"] == 1].iloc[0]
_base_text    = str(_sample_buyer["review_text"])
_base_title   = str(_sample_buyer["review_title"])

# (id, description, title, rating, review_text, price, expected_label)
EDGE_CASES = [
    # ── GROUP A: Input preprocessing ──────────────────────────────────────
    (1,  "Very short text – no tokens survive, rating dominates",
     "ok", 5.0, "ok", PRICE_BUDGET, 1),
    (2,  "Very long text (1000+ words)",
     "Amazing moisturiser", 5.0,
     ("This moisturiser is absolutely incredible and my skin glows every day. " * 30).strip(),
     PRICE_MID, 1),
    (3,  "ALL CAPS positive",
     "BEST PRODUCT EVER", 5.0,
     "THIS PRODUCT IS AMAZING. SKIN FEELS SO SMOOTH AND HYDRATED. WOULD DEFINITELY RECOMMEND.",
     PRICE_MID_LOW, 1),
    (4,  "ALL CAPS negative",
     "WORST PRODUCT EVER", 1.0,
     "THIS PRODUCT IS TERRIBLE. IT BROKE ME OUT AND CAUSED IRRITATION. COMPLETE WASTE OF MONEY.",
     PRICE_MID, 0),
    (5,  "Repeated characters – normalisation",
     "Sooooo goooood", 5.0,
     "I loooove this product soooo much. My skin feels amaaaazing after using it.",
     PRICE_MID_HIGH, 1),
    (6,  "HTML entities in text",
     "Great &amp; effective", 5.0,
     "This product is great &amp; effective. My skin looks &lt;perfect&gt; after using it.",
     PRICE_PREMIUM, 1),
    (7,  "Emoji-heavy text",
     "Love it ❤️", 5.0,
     "❤️\U0001f31f✨ This product is amazing \U0001f495 my skin glows \U0001f338 highly recommend \U0001f44d",
     PRICE_MID, 1),
    (8,  "URLs in text",
     "Good product", 4.0,
     "Great product, learned about it from www.skincareaddicts.com and https://reddit.com/r/SkincareAddiction. Works well.",
     PRICE_MID_LOW, 1),
    (9,  "@mentions and #hashtags",
     "#skincare fave", 5.0,
     "@brand this is my fave product! #skincare #beauty #glowup the results are incredible.",
     PRICE_BUDGET, 1),
    (10, "Digits only – all tokens stripped, rating dominates",
     "12345", 5.0, "12345 5 100 200", PRICE_MID, 1),
    (11, "Punctuation only text",
     "...!!!", 5.0, "!!! ??? ... !!!", PRICE_MID, 1),
    (12, "Single word review",
     "Excellent", 5.0, "Excellent", PRICE_MID_LOW, 1),
    (13, "CamelCase text",
     "GoodProduct", 4.0,
     "ThisProductIsSoGoodMyFaceFeelsSoSmoothAndHydratedAfterEveryUse.",
     PRICE_MID, 1),
    (14, "Text with excessive apostrophes",
     "It's the best", 4.0,
     "I've been using this for months and it's completely transformed my skin. It's lightweight, it's effective, and it's affordable.",
     PRICE_MID_LOW, 1),
    (15, "Mixed English and transliteration",
     "Bahut achha hai", 5.0,
     "This product is bahut achha. Maine use kiya aur skin glow kar rahi hai. Highly recommend karta hoon.",
     PRICE_MID, 1),
    # ── GROUP B: Rating vs text contradictions ────────────────────────────
    (16, "Rating 5★, strongly negative text",
     "Worst product ever", 5.0,
     "This is the worst product I have ever used. It broke me out badly, caused severe irritation, "
     "smells terrible, and is a complete waste of money. I want a refund.",
     PRICE_MID_LOW, 0),
    (17, "Rating 1★, strongly positive text",
     "Amazing product love it", 1.0,
     "This product is absolutely amazing. My skin has never looked better. "
     "Smooth, hydrated, glowing. Highly recommend to everyone.",
     PRICE_MID, 1),
    (18, "Rating 5★, negative title, positive body text",
     "Terrible, do not buy", 5.0,
     "I am so impressed with this moisturiser. My skin feels incredibly soft and hydrated. "
     "The texture is light and absorbs quickly. Will definitely repurchase.",
     PRICE_MID_HIGH, 1),
    (19, "Rating 1★, positive title, negative body text",
     "Best serum I have ever tried", 1.0,
     "This product is absolutely terrible. It broke me out, smells awful, "
     "and did nothing for my skin. Complete waste of money. Returning immediately.",
     PRICE_PREMIUM, 0),
    (20, "Rating 2★, mildly negative text",
     "Not impressed", 2.0,
     "This product did not work for me. My skin felt dry and tight after using it. "
     "I expected much better results given the price.",
     PRICE_MID_LOW, 0),
    (21, "Rating 4★, mildly negative text",
     "Good but not great", 4.0,
     "It works okay but I am not totally impressed. Skin feels slightly better but nothing dramatic. "
     "Probably would not repurchase.",
     PRICE_MID, 0),
    (22, "Rating 3★, strongly positive text",
     "Decent product", 3.0,
     "This moisturiser has completely transformed my skin. I wake up every morning with a glow. "
     "My friends keep asking what I use. Absolutely love it and will keep buying.",
     PRICE_MID, 1),
    (23, "Rating 3★, strongly negative text",
     "Average at best", 3.0,
     "This product is absolutely horrible. Caused severe breakouts, irritation, and redness. "
     "Smells chemical and feels heavy on the skin. Would never buy this again.",
     PRICE_MID_LOW, 0),
    (24, "Rating 5★, sarcastic negative title and body",
     "Oh wow so amazing (not)", 5.0,
     "I gave five stars because I cannot give zero. This product destroyed my skin. "
     "Caused cystic acne and left scars. Do not buy under any circumstances.",
     PRICE_MID, 0),
    (25, "Rating 4★, detailed positive with one complaint",
     "Great but overpriced", 4.0,
     "This serum genuinely works. My dark spots have faded, skin feels plump, and the texture is wonderful. "
     "Only downside is the price, but the results justify it. Will repurchase.",
     PRICE_PREMIUM, 1),
    (26, "Rating 2★, positive opening then negative overall",
     "Started well, then disappointed", 2.0,
     "The first week was great, my skin felt soft and hydrated. "
     "But after two weeks I started breaking out badly. The results did not last.",
     PRICE_MID_HIGH, 0),
    (27, "Rating 5★, short negative text",
     "Did not work", 5.0,
     "Completely useless. Waste of money.",
     PRICE_BUDGET, 0),
    (28, "Rating 1★, short positive text",
     "Love it", 1.0, "Great product, works well.",
     PRICE_MID_LOW, 1),
    (29, "Rating 3★, neutral factual text",
     "Does what it says", 3.0,
     "This moisturiser is lightweight and absorbs well. Provides adequate hydration for daily use. "
     "Nothing special but does the job.",
     PRICE_MID, 1),
    (30, "Rating 5★, ALL CAPS negative text",
     "DO NOT BUY", 5.0,
     "DO NOT BUY THIS PRODUCT. IT RUINED MY SKIN. CAUSED BREAKOUTS AND REDNESS. COMPLETE SCAM.",
     PRICE_MID_HIGH, 0),
    # ── GROUP C: Price contradictions ─────────────────────────────────────
    (31, "High price + low rating + bad review – all signals Not Buyer",
     "Not worth the price", 1.0,
     "Extremely overpriced and completely ineffective. Caused breakouts and irritation. "
     "Expected much better. Very disappointed and will not repurchase.",
     _max_price, 0),
    (32, "Low price + high rating + great review – all signals Buyer",
     "Amazing value for money", 5.0,
     "I cannot believe how good this product is for the price. "
     "My skin looks brighter, feels softer, and results are better than expensive alternatives.",
     _min_price, 1),
    (33, "High price + high rating + bad review – conflict",
     "Disappointing for the price", 5.0,
     "For this amount of money I expected a miracle, but this product did nothing special. "
     "Skin feels greasy, broke me out, and the smell is overpowering. Not worth it at all.",
     _max_price, 0),
    (34, "Low price + low rating + positive review – conflict",
     "Surprised by this cheap find", 1.0,
     "I bought this on a whim because it was so affordable and honestly it is one of the best "
     "skincare products I have ever used. My skin glows, feels hydrated, and looks younger.",
     _min_price, 1),
    (35, "Premium price + 5★ + detailed positive – aligned Buyer",
     "Worth every rupee", 5.0,
     "At this price point I expected excellence and that is exactly what I got. "
     "This serum has visibly reduced my fine lines, evened my skin tone, and given me a natural glow. "
     "Worth every penny and I will repurchase.",
     PRICE_PREMIUM, 1),
    (36, "Budget price + 1★ + detailed negative – aligned Not Buyer",
     "Cheap and useless", 1.0,
     "For this price you get what you pay for. This moisturiser is watery, has no effect, "
     "and actually made my skin worse. Packaging is flimsy and the smell is off-putting.",
     PRICE_BUDGET, 0),
    (37, "Mid-high price + 4★ + positive text",
     "Good investment for skin", 4.0,
     "This product is pricey but the quality shows. My skin has improved noticeably over three weeks. "
     "The texture is luxurious and a little goes a long way. Happy with the purchase.",
     PRICE_MID_HIGH, 1),
    (38, "Mid price + 2★ + negative text",
     "Not worth it", 2.0,
     "I was excited to try this based on reviews but it did absolutely nothing for my skin. "
     "After a full month there is no improvement. The scent is overpowering and it pills under makeup.",
     PRICE_MID, 0),
    (39, "Budget price + 5★ + very positive text",
     "Best budget moisturiser", 5.0,
     "I have tried products ten times this price and this is better. "
     "Skin feels soft, hydrated, and looks healthy. My go-to daily moisturiser now.",
     PRICE_BUDGET, 1),
    (40, "Premium price + 1★ + extremely negative text",
     "Expensive garbage", 1.0,
     "I paid a premium price for what turned out to be the worst product I have used. "
     "Caused immediate irritation, redness, and a full breakout. The brand should be ashamed.",
     PRICE_PREMIUM, 0),
    (41, "Mid-low price + 3★ + neutral text",
     "Average product", 3.0,
     "It is an okay moisturiser. Nothing spectacular but nothing terrible either. "
     "Does its basic job of hydrating skin. Would consider repurchasing.",
     PRICE_MID_LOW, 1),
    (42, "Mid-high price + 5★ + positive value text",
     "Quality justifies the price", 5.0,
     "Yes it is expensive but my skin has never looked better. "
     "The formula is rich without being heavy and the results are visible within a week. "
     "I consider this an investment in my skin.",
     PRICE_MID_HIGH, 1),
    (43, "Max price + 3★ + mixed expectations",
     "Mixed feelings about the price", 3.0,
     "The product itself is decent and does hydrate well. "
     "But at this price I expected transformative results, not just average ones. "
     "Would only recommend if on sale.",
     _max_price, 1),
    (44, "Min price + 4★ + positive text",
     "Pleasantly surprised", 4.0,
     "Bought this expecting nothing much given the price and was genuinely surprised. "
     "Good hydration, pleasant smell, absorbs quickly. Will definitely buy again.",
     _min_price, 1),
    (45, "Mid price + 1★ + allergic reaction",
     "Caused allergic reaction", 1.0,
     "This product caused a severe allergic reaction. My face swelled up and broke out in hives. "
     "I had to see a dermatologist. Would give zero stars if I could.",
     PRICE_MID, 0),
    # ── GROUP D: Title vs text contradictions ─────────────────────────────
    (46, "Positive title, negative detailed text",
     "Loved this product", 2.0,
     "I wanted to love this product but it just does not work for my skin. "
     "Heavy, greasy, caused breakouts. The ingredients look great on paper but did not deliver.",
     PRICE_MID_LOW, 0),
    (47, "Negative title, positive detailed text",
     "Terrible product don't buy", 5.0,
     "My skin has completely transformed since using this. Pores look smaller, skin tone evened out, "
     "and I get compliments daily. This is now a permanent part of my routine.",
     PRICE_MID, 1),
    (48, "Neutral title, strongly positive text",
     "Review", 5.0,
     "This is hands down the best moisturiser I have ever used. Rich, creamy, non-greasy, "
     "and my skin looks amazing. I have repurchased three times already.",
     PRICE_MID_HIGH, 1),
    (49, "Neutral title, strongly negative text",
     "Review", 1.0,
     "Absolutely terrible. Destroyed my skin barrier, caused cystic acne, and felt like plastic on the face. "
     "I threw the bottle away after three uses.",
     PRICE_MID, 0),
    (50, "Positive title, complaint is about shipping not product",
     "Amazing moisturiser", 1.0,
     "The product looks fine but delivery took six weeks, packaging was completely destroyed, "
     "and customer service refused to help. Will never order from this seller again.",
     PRICE_MID_HIGH, 0),
    (51, "Negative title, complaint is about shipping not product",
     "Terrible experience", 3.0,
     "The moisturiser itself works well and my skin feels great. "
     "Docking stars only because the delivery was very delayed and packaging arrived dented.",
     PRICE_MID, 1),
    (52, "Title mentions competitor, text is positive",
     "Better than Clinique", 5.0,
     "I have been using the Clinique moisturiser for years but this is genuinely better. "
     "More hydrating, better texture, and works out cheaper per use. Very impressed.",
     PRICE_MID_LOW, 1),
    (53, "Title says gift but text reviews the product",
     "Bought as gift but tried it too", 4.0,
     "I bought this as a birthday gift for my sister and ended up trying it myself. "
     "The serum is lovely, skin feels soft and looks dewy. Will buy for myself next time.",
     PRICE_MID, 1),
    (54, "Title says received as gift – not purchased",
     "Received as gift, not purchased", 5.0,
     "I received this as a gift and have been using it daily. The results are fantastic. "
     "Skin feels incredibly soft and my dark spots have faded significantly.",
     PRICE_MID_HIGH, 0),
    (55, "Title is a question, text is a genuine review",
     "Does this really work?", 4.0,
     "After trying this for a month I can confirm it absolutely does work. "
     "My skin texture has improved and I get regular compliments. Very pleased.",
     PRICE_MID_LOW, 1),
    # ── GROUP E: Mixed signals ────────────────────────────────────────────
    (56, "Loves texture but broke out with cystic acne",
     "Love it but broke me out", 3.0,
     "I really love the texture and the smell of this product, it absorbs beautifully. "
     "However it completely broke me out and caused painful cystic acne. "
     "I want to love it but my skin just cannot tolerate it.",
     PRICE_MID_HIGH, 0),
    (57, "Great for oily skin, bad for dry skin",
     "Depends on your skin type", 3.0,
     "If you have oily skin this is brilliant. Controls shine and keeps skin clear. "
     "But I have dry skin and this made my face feel tight and flaky. Not universally suitable.",
     PRICE_MID, 1),
    (58, "Great results initially, then stopped working",
     "Great at first, then stopped working", 2.0,
     "The first two weeks were incredible. Skin was glowing and hydrated. "
     "But by week three the effects wore off and my skin returned to normal. "
     "Disappointing for the price.",
     PRICE_MID_HIGH, 0),
    (59, "Good product, would repurchase but only at lower price",
     "Good product, overpriced", 3.0,
     "This moisturiser does work. Skin feels hydrated and looks healthy. "
     "But I would only repurchase if it were at least thirty percent cheaper. "
     "There are better value alternatives.",
     PRICE_PREMIUM, 1),
    (60, "Amazing results but terrible smell",
     "Amazing but smells awful", 4.0,
     "The results are honestly incredible. My skin has never looked better. "
     "But the smell is absolutely terrible – strong chemical odour that lingers all day.",
     PRICE_MID, 1),
    (61, "Product is fine but packaging broke",
     "Product ok, packaging awful", 3.0,
     "The cream itself is decent and does what it claims. "
     "But the pump dispenser broke after two weeks and the cap keeps falling off. "
     "Frustrating for the price.",
     PRICE_MID_LOW, 1),
    (62, "Loved the sample, disappointed with full size",
     "Sample was better than full size", 2.0,
     "I tried the sample sachet and loved it. The full size product feels completely different. "
     "Heavier texture, different smell, less effective. Suspect reformulation.",
     PRICE_PREMIUM, 0),
    (63, "Positive overall but mentions initial purging",
     "Works well but caused initial breakouts", 4.0,
     "This serum has improved my skin texture and reduced my dark spots noticeably. "
     "I did experience some initial purging in the first two weeks "
     "but they cleared and the results since then have been great.",
     PRICE_MID, 1),
    (64, "Good but not for sensitive skin",
     "Good but not for sensitive skin", 3.0,
     "This product works well for my normal to oily skin. "
     "However my sister with sensitive skin had a bad reaction. "
     "Good product but be careful if you have reactive skin.",
     PRICE_MID_LOW, 1),
    (65, "Before and after style positive review",
     "Transformed my skin in 30 days", 5.0,
     "Before: dull, dry, patchy skin with visible pores. "
     "After 30 days: glowing, hydrated, smooth with barely visible pores. "
     "This product genuinely delivers on its promises.",
     PRICE_MID_HIGH, 1),
    # ── GROUP F: Review content types ─────────────────────────────────────
    (66, "Received free sample – not purchased",
     "Free sample review", 5.0,
     "I received a free sample of this product and have been using it for a week. "
     "My skin feels amazing and I am definitely going to purchase the full size.",
     PRICE_MID, 0),
    (67, "Compares favourably to luxury brand",
     "Better than La Mer", 5.0,
     "I have used La Mer for years and this is honestly comparable at a fraction of the price. "
     "The hydration is incredible and my skin looks just as good. Switching permanently.",
     PRICE_PREMIUM, 1),
    (68, "Dermatologist recommended product",
     "Dermatologist recommended and approved", 5.0,
     "My dermatologist recommended this serum for my hyperpigmentation. "
     "After eight weeks my skin tone is significantly more even. Very happy.",
     PRICE_MID_HIGH, 1),
    (69, "Safe during pregnancy – positive",
     "Safe during pregnancy", 5.0,
     "This moisturiser is free of retinol and salicylic acid so it is safe during pregnancy. "
     "It keeps my skin hydrated and the formula is gentle. Used throughout my pregnancy.",
     PRICE_MID, 1),
    (70, "Returned for refund – Not Buyer outcome",
     "Returned for refund", 1.0,
     "I returned this product for a refund after one week of use. "
     "It caused immediate irritation and my skin became red and inflamed.",
     PRICE_MID_LOW, 0),
    (71, "Review about smell only – positive",
     "Smells incredible", 5.0,
     "I bought this primarily because of the reviews about the scent and it lives up to the hype. "
     "It smells absolutely divine. The skincare results are also good but the fragrance alone is worth it.",
     PRICE_BUDGET, 1),
    (72, "Review about texture and application only",
     "Perfect texture", 5.0,
     "The texture of this moisturiser is absolutely perfect. "
     "It melts into the skin immediately, leaves no residue, and sits beautifully under makeup.",
     PRICE_MID, 1),
    (73, "Severe allergic reaction – discontinued",
     "Caused severe reaction, discontinued", 1.0,
     "After three days of use I developed severe contact dermatitis. "
     "My face was swollen, red, and incredibly painful. "
     "I discontinued immediately and needed prescribed steroid cream to recover.",
     PRICE_MID_HIGH, 0),
    (74, "Very generic short positive review",
     "Good product", 5.0,
     "Good product. Works well. Happy with the purchase. Would recommend.",
     PRICE_MID_LOW, 1),
    (75, "Very generic short negative review",
     "Bad product", 1.0,
     "Bad product. Did not work. Waste of money. Would not recommend.",
     PRICE_MID, 0),
    (76, "Loyal repurchaser on third bottle",
     "On my third bottle", 5.0,
     "This is my third bottle of this moisturiser and I have no intention of stopping. "
     "Consistent results, great texture, and my skin has never been better.",
     PRICE_MID, 1),
    (77, "Bought as a birthday gift for someone else",
     "Bought as a birthday gift", 4.0,
     "I bought this for my mother's birthday and she absolutely loves it. "
     "Her skin has improved noticeably and she keeps asking me to reorder. Great gift option.",
     PRICE_MID_HIGH, 1),
    (78, "Conditional positive – would be perfect if cheaper",
     "Would be perfect if cheaper", 4.0,
     "This product genuinely works and I have seen real improvement in my skin. "
     "The formula feels premium and the results are visible. "
     "If it were thirty percent cheaper I would give it five stars without hesitation.",
     PRICE_PREMIUM, 1),
    (79, "Expresses purchase regret",
     "Wish I had not bought this", 1.0,
     "I spent a lot of money on this product based on influencer recommendations "
     "and deeply regret it. It did nothing and caused mild irritation. "
     "I feel completely misled by the marketing.",
     PRICE_PREMIUM, 0),
    (80, "Perfect for sensitive skin",
     "Perfect for sensitive skin", 5.0,
     "I have extremely sensitive skin that reacts to almost everything. "
     "This moisturiser is the first product in years that has not caused any reaction. "
     "Gentle, effective, and my skin loves it.",
     PRICE_MID_LOW, 1),
    # ── GROUP G: Linguistic variations ────────────────────────────────────
    (81, "Non-ASCII – French positive review",
     "Très bien", 4.0,
     "Ce produit est très bien. Ma peau est douce et hydratée. Je le recommande vivement.",
     PRICE_MID_LOW, 1),
    (82, "Very formal – academic writing style",
     "A thorough assessment", 4.0,
     "Upon application this emollient demonstrates adequate occlusive properties. "
     "The humectant components provide satisfactory transepidermal water loss prevention. "
     "Clinical-grade hydration is observed within the first application cycle.",
     PRICE_MID_HIGH, 1),
    (83, "Very informal – Gen-Z slang",
     "no cap this is fire", 5.0,
     "bestie this moisturiser is absolutely it no cap. my skin is literally glowing slay. "
     "hits different from other brands fr fr. would eat it if i could periodt.",
     PRICE_MID, 1),
    (84, "Review written as pros and cons list",
     "Pros and cons review", 4.0,
     "Pros: great texture, absorbs fast, visible results in one week, lovely smell. "
     "Cons: expensive, small bottle, slightly sticky if you use too much. "
     "Overall: would recommend and will repurchase.",
     PRICE_MID_HIGH, 1),
    (85, "Review with spelling mistakes throughout",
     "Bset prodcut evr", 5.0,
     "I absoltuely lov this moituriser. My skiN feels so goood and smoohth after useing it. "
     "The textur is amazng and it absorbs so quikly. Definately reccomend to evreyone.",
     PRICE_MID_LOW, 1),
    (86, "Review written entirely as questions",
     "Is this even tested?", 1.0,
     "Why does this smell so chemical? How did this pass quality control? "
     "Who approved this formula? Why is it so greasy? Why did I waste my money on this?",
     PRICE_MID, 0),
    (87, "Review mentioning full skincare routine layering",
     "Great addition to my routine", 5.0,
     "I use this after my vitamin C serum and before my SPF and it layers perfectly. "
     "No pilling, no reaction, and it actually enhances the other products. "
     "A fantastic addition to any skincare routine.",
     PRICE_MID, 1),
    (88, "Review with only positive adjectives – no sentences",
     "Perfect hydrating effective", 5.0,
     "Hydrating. Lightweight. Effective. Fragrant. Non-greasy. Smooth. Plumping. Brightening. Recommended.",
     PRICE_MID_LOW, 1),
    (89, "Review with only negative adjectives – no sentences",
     "Greasy heavy irritating", 1.0,
     "Greasy. Heavy. Sticky. Irritating. Smelly. Ineffective. Overpriced. Disappointing. Pore-clogging. Terrible.",
     PRICE_MID, 0),
    (90, "Duplicate of buyer baseline review",
     _base_title, 5.0, _base_text, PRICE_PREMIUM, 1),
    # ── GROUP H: Real-world skincare scenarios ────────────────────────────
    (91, "Sunscreen – no white cast, repurchased 4 times",
     "Best SPF for daily use", 5.0,
     "This sunscreen is the first I have found that does not leave a white cast. "
     "Lightweight, no greasy residue, and my skin has not tanned at all this summer. "
     "Repurchased four times already.",
     PRICE_MID_LOW, 1),
    (92, "Foundation – wrong shade, bad service",
     "Wrong shade, terrible service", 1.0,
     "The foundation was completely the wrong shade and oxidised within an hour. "
     "Customer service refused to exchange it. "
     "The formula itself may be fine but I will never buy from this brand again.",
     PRICE_MID, 0),
    (93, "Vitamin C serum – dark spots faded in 4 weeks",
     "Dark spots faded in 4 weeks", 5.0,
     "I have been struggling with post-acne marks for years. "
     "After four weeks of using this vitamin C serum my dark spots have faded by at least fifty percent. "
     "The results are visible and I am genuinely impressed.",
     PRICE_MID_HIGH, 1),
    (94, "Retinol – purging phase then positive results",
     "Purging phase but worth it", 4.0,
     "The first three weeks caused some purging breakouts which was expected with retinol. "
     "After week four my skin started to clear and now at week eight it looks incredible. "
     "Smooth, even, and glowing. Patience pays off.",
     PRICE_MID, 1),
    (95, "Lip balm – repurchased 6 times",
     "Best lip care I have tried", 5.0,
     "This lip balm is absolutely incredible. Lips feel healed after the first use. "
     "The formula is nourishing, not sticky, and lasts all day. I have repurchased six times.",
     PRICE_BUDGET, 1),
    (96, "Toner – dried out and damaged skin barrier",
     "Dried out my skin", 1.0,
     "This toner completely stripped my skin of moisture. "
     "After two weeks my face was dry, flaky, and red. "
     "It disrupted my skin barrier and it took months to repair.",
     PRICE_MID_LOW, 0),
    (97, "Eye cream – reduced dark circles in 2 weeks",
     "Reduced dark circles in 2 weeks", 5.0,
     "I have had dark circles my whole life and tried numerous products with no results. "
     "This eye cream has visibly lightened them in just two weeks. "
     "Also reduced puffiness significantly. A genuine miracle product.",
     PRICE_MID_HIGH, 1),
    (98, "Moisturiser – first product that does not cause breakouts",
     "Finally a moisturiser that does not break me out", 5.0,
     "I have acne-prone skin and finding a non-comedogenic moisturiser has been a nightmare. "
     "This is the first product in years that has not caused a single breakout. "
     "Skin is hydrated and calm. Life changing.",
     PRICE_MID, 1),
    (99, "Anti-ageing cream – no visible results after 3 months",
     "No visible anti-ageing results", 2.0,
     "I used this anti-ageing cream religiously for three months and saw absolutely no difference. "
     "My fine lines are exactly the same. "
     "For this price I expected at least some visible improvement.",
     PRICE_PREMIUM, 0),
    (100, "Micellar water – removes waterproof makeup in one swipe",
     "Removes everything including waterproof mascara", 5.0,
     "This micellar water removes my full face of makeup including waterproof mascara in one swipe. "
     "No rubbing, no irritation, and my skin feels clean without being stripped. "
     "The best makeup remover I have ever used.",
     PRICE_MID_LOW, 1),
]

# Ground-truth labels for the 100 edge cases
y_edge = np.array([case[6] for case in EDGE_CASES])
print(f"Edge cases: {len(EDGE_CASES)}  |  Buyers: {y_edge.sum()}  |  Not Buyers: {(y_edge==0).sum()}")


 ### 2.5 Head A — Pre-trained FastText + LightGBM on `review_text`, `review_title`, `rating`, `price`



 Uses the pre-trained model from

 `app/nlp/review_text_title_rating_price_model/buyer_pipeline_text_numeric.py`.

 Evaluated row-by-row over the 100 EDGE_CASES.

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from app.nlp.review_text_title_rating_price_model.buyer_pipeline_text_numeric import predict as buyer_text_numeric_predict

p_buyer_text_numeric = np.array([
    buyer_text_numeric_predict(
        case[4],   # review_text
        case[2],   # review_title
        case[3],   # rating
        case[5],   # price
    )[1][1]
    for case in EDGE_CASES
])
yhat_text_numeric = (p_buyer_text_numeric >= 0.5).astype(int)
print(f"Head A (text+title+rating+price) : acc={accuracy_score(y_edge, yhat_text_numeric):.4f}  "
      f"F1={f1_score(y_edge, yhat_text_numeric):.4f}")


 ### 2.6 Head B — Pre-trained FastText + LightGBM on `review_text`, `review_title`



 Uses the pre-trained model from

 `app/nlp/review_text_title_model/buyer_pipeline.py`.

 Evaluated row-by-row over the 100 EDGE_CASES.

In [ ]:
from app.nlp.review_text_title_model.buyer_pipeline import predict as buyer_title_text_predict

p_buyer_title_text = np.array([
    buyer_title_text_predict(
        case[4],   # review_text
        case[2],   # review_title
    )[1][1]
    for case in EDGE_CASES
])
yhat_title_text = (p_buyer_title_text >= 0.5).astype(int)
print(f"Head B (text+title) : acc={accuracy_score(y_edge, yhat_title_text):.4f}  "
      f"F1={f1_score(y_edge, yhat_title_text):.4f}")


 ### 2.7 Head C — XGBoost on `[rating, log1p(price)]`



 Two columns: `rating` (1–5) and `log1p(price)`. Our Milestone 1

 study showed that adding `rating` and `price` improved performance

 over text-only models; Head C captures that numeric signal directly.

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def _numeric_features(rating, price):
    r = np.asarray(rating, dtype=float).reshape(-1, 1)
    lp = np.log1p(np.asarray(price, dtype=float)).reshape(-1, 1)
    return np.hstack([r, lp])

X_num_all = _numeric_features(nykaa["review_rating"], nykaa["price"])

numeric_model = Pipeline([
    ("clf", XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

numeric_model.fit(X_num_all, y)

X_num_edge = _numeric_features(
    [case[3] for case in EDGE_CASES],   # rating
    [case[5] for case in EDGE_CASES],   # price
)
p_num = numeric_model.predict_proba(X_num_edge)[:, 1]

yhat_num = (p_num >= 0.5).astype(int)

print(
    f"Head C (rating+price) : acc={accuracy_score(y_edge, yhat_num):.4f}  "
    f"F1={f1_score(y_edge, yhat_num):.4f}"
)


 ### 2.8 Fusion — mean of `predict_proba`, threshold 0.5



 The fused probability is the unweighted arithmetic mean of the three

 heads' `predict_proba` outputs. We threshold at 0.5 for the final

 label.



 A 4-cell grid of confusion matrices below makes the diversity of

 errors visible — each head misclassifies a different slice of the

 test set, and averaging cancels some of those errors.

In [ ]:
W_TEXT_NUMERICAL = 0.70
W_TEXT           = 0.10
W_NUM            = 0.20

_w_sum = W_TEXT_NUMERICAL + W_TEXT + W_NUM
p_fused = (
    W_TEXT_NUMERICAL * p_buyer_text_numeric +
    W_TEXT           * p_buyer_title_text   +
    W_NUM            * p_num
) / _w_sum
y_hat = (p_fused >= 0.5).astype(int)

metrics = pd.DataFrame({
    "model": ["Head A — text+title+rating+price", "Head B — text+title", "Head C — rating+price", "Fused"],
    "accuracy": [
        accuracy_score(y_edge, yhat_text_numeric),
        accuracy_score(y_edge, yhat_title_text),
        accuracy_score(y_edge, yhat_num),
        accuracy_score(y_edge, y_hat),
    ],
    "F1": [
        f1_score(y_edge, yhat_text_numeric),
        f1_score(y_edge, yhat_title_text),
        f1_score(y_edge, yhat_num),
        f1_score(y_edge, y_hat),
    ],
}).round(4)
print(metrics.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for ax, (name, yhat) in zip(axes.flat, [
    ("Head A — text+title+rating+price", yhat_text_numeric),
    ("Head B — text+title",             yhat_title_text),
    ("Head C — rating+price",           yhat_num),
    ("Fused",                  y_hat),
]):
    cm = confusion_matrix(y_edge, yhat)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
plt.show()


 ### 2.9 Head A probability distribution — sanity check



 Distribution of P(Buyer) from Head A across the 100 edge cases.

 Buyers should cluster near 1.0, non-buyers near 0.0.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(p_buyer_text_numeric[y_edge == 1], bins=40, alpha=0.6, label="Buyer (1)")
ax.hist(p_buyer_text_numeric[y_edge == 0], bins=40, alpha=0.6, label="Not Buyer (0)")
ax.set_xlabel("P(Buyer) — Head A (text+title+rating+price)")
ax.set_ylabel("count")
ax.set_title("Head A (text+title+rating+price) — predicted probability distribution")
ax.legend()
fig.tight_layout()
plt.show()


 ### Analysis



 - **Head A (text+title+rating+price):** the strongest head — it sees

   all available signals. FastText embeddings capture subword semantics

   and handle OOV skincare vocabulary; LightGBM handles the mixed

   text+numeric feature space well.

 - **Head B (text+title):** weaker than A because it lacks the

   numeric signal, but its errors are somewhat independent of A's,

   which is the key property for fusion benefit. Its low weight (0.10)

   reflects this.

 - **Head C (rating+price):** XGBoost on `[rating, log1p(price)]`

   is surprisingly strong because `rating=5` is highly correlated

   with `is_a_buyer=1` by construction. Contributes complementary

   signal when the text heads are uncertain.

 - **Fused:** the weighted combination (0.70 / 0.10 / 0.20) pushes

   F1 above any single head's F1, confirming that the heads'

   errors are not perfectly correlated.



 ### Limitations



 - **Heads A and B are pre-trained on the full corpus.** Evaluation

   on the 100 edge cases is out-of-sample by construction, but any

   review that appeared verbatim in the Nykaa training corpus would

   be in-sample for those heads.

 - **Head C trains on the full corpus evaluated on edge cases.** The

   edge cases use synthetic price tiers, so there is no overlap with

   training data for Head C's numeric features.

 - **The numeric head can be overconfident on sarcastic reviews.**

   A reviewer giving 5 stars ironically gets a strong "buyer" signal

   from Head C; the text heads partially compensate, but the 0.70

   weight on Head A is the main safeguard.

 - **Threshold is hard-coded at 0.5.** A small calibration step on

   held-out data could shift the decision boundary without retraining.



 ### App pointer



 `app/nlp/classifier.py:ReviewClassifier` ships this exact

 architecture to the live Flask app. HEAD A and HEAD B are loaded from

 pre-trained artefacts in `app/nlp/review_text_title_rating_price_model/`

 and `app/nlp/review_text_title_model/`. HEAD C (`numeric_model.joblib`)

 is trained by `ReviewClassifier.train()` and saved via

 `ReviewClassifier.save()`. The Flask route `POST /product/<id>/review`

 (`app/__init__.py`) calls `classifier.predict(title, rating, text,

 price)` and shows the result to the reviewer for optional override.



 ### Milestone 1 pointer (detailed)



 - **Tokenizer + filters** (regex, lowercase, length ≥ 2, stopwords):

   `knowledge/milestone1_task1.ipynb` §1.2.3 ("Cleaning Pipeline &

   Tokenisation") and §1.2.4 ("Post-Tokenisation Preprocessing").

 - **FastText embeddings + LightGBM:** our Milestone 1 study

   (`knowledge/milestone1_task2_3.ipynb`) established FastText+LightGBM

   as the top-performing combination; Heads A and B reuse those trained

   artefacts directly.

 - **What the pre-trained models cover:** Head A's model also includes

   `rating` and `price` alongside the FastText embedding, consistent

   with M1's finding that numeric metadata improves classification.

 ## 3 Task 3 - Similar-Item Recommendations



 ### Problem framing



 The Milestone 2 spec for Task 3: *"the system should allow a customer

 to select a specific item, after which it will automatically display

 a set of similar items. You are required to define an appropriate

 similarity measure and compute the similarity between items (can use

 the vector representations e.g., text features, embeddings and/or

 other relevant information sources)."*



 For **Task 3**. It covers,

 in order:



 1. **Requirement checklist** - every clause of the Task 3 spec, with

    the place in this notebook (or the app) where the clause is met.

 2. **Data used for the recommender** - exactly which columns of

    `products.csv` feed the similarity index, and why.

 3. **Pipeline overview** - the end-to-end flow from "customer opens

    a product page" to "six recommended cards render".

 4. **Similarity score formula** - the math, written first as a

    formula and then in plain words.

 5. **Recommender code, with inline comments** - the same function

    that ships to the live app, annotated line by line.

 6. **Worked example: before vs after the category boost** - the

    actual top-6 table for a probe product, both with and without the

    +0.1 same-category adjustment, so the boost's effect is visible.

 7. **Efficiency and scalability** - why this approach is fast enough

    to run on every product-detail page render, and how it scales.

 8. **App integration** - exactly where on the live website Task 3's

    output appears (route, template, layout slot).

 9. **Conclusion** - what Task 3 delivers, where it could go next.





 ## 3.1 Requirement checklist



 The checklist below maps every clause of the spec + rubric to the

 section of this notebook (or the file in the app) that satisfies it.

 Every row is checked.



 | # | Spec / rubric clause | Where it is met |

 |---|---|---|

 | ✓ 1 | *"allow a customer to select a specific item"* | The Flask route `GET /product/<id>` is the per-item selection. Section 8 of this notebook. |

 | ✓ 2 | *"automatically display a set of similar items"* | The product-detail page calls `SearchIndex.similar(product_id, top_n=6)` during render and lays the six cards out in a grid. Sections 8 and 9. |

 | ✓ 3 | *"define an appropriate similarity measure"* | **Cosine similarity** over a char-n-gram TF-IDF representation of `product_name + " " + category`, plus a category boost. Sections 3 and 4. |

 | ✓ 4 | *"compute the similarity between items"* | Section 5 (the code) computes one row of the pairwise similarity matrix on demand. Section 6 shows the resulting top-6 list for a probe product. |

 | ✓ 5 | *"use … vector representations … text features"* | Char-n-gram TF-IDF vectors. Section 2 explains the data; Section 4 explains the vectorisation. |

 | ✓ 6 | Recommendations are **clearly displayed** | Section 9 documents each card's contents (image, category, name, price) and the section heading "Similar items · Ranked by similarity". |

 | ✓ 7 | Method is **justifiable, proper, effective** (rubric language) | Section 3 (pipeline), Section 4 (formula), Section 7 (efficiency) lay out the justification. Section 6 shows it working. |





 In the live app, opening any product-detail page

 (`GET /product/<id>`) shows six recommended products below the main

 content. The recommendations should feel "related" to the customer —

 same kind of product, similar audience.





 ## 3.2 Data used for the recommender



 The recommender uses **only the product catalogue** — no reviews, no

 user history, no clicks.



 ### Why not use the reviews?



 About a third of the 1,000-product catalogue has zero reviews in

 `reviews.csv`. A reviews-based similarity (e.g. averaged-review

 embeddings) would **cold-start fail** for exactly the products that

 need recommendations the most — the long-tail items the customer

 hasn't found yet. Sticking to `product_name + category` gives the

 recommender uniform coverage across the whole catalogue. This also avoids giving an unfair advantage to popular products with many reviews while under-representing newer or less-reviewed products.



 ### Why join `product_name` and `category`



 Two short strings concatenated:



 ```

 "Maybelline SuperStay Matte Ink Liquid Lipstick" + " " + "Makeup"

 ```



 become a single document that the vectorizer encodes. The category

 contributes a few extra char-n-grams that reinforce the kind of

 product, which makes the vectors of two "Makeup" items lean towards

 each other a little even if their names share few n-grams.

 ## Setup



 A single import block + the CSV load. No randomness - the recommender

 is fully deterministic given the product catalogue.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# This notebook lives at knowledge/milestone2_task3.ipynb, so the
# parent of the notebook's directory is the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "knowledge" else Path.cwd()
KNOWLEDGE_DIR = REPO_ROOT / "knowledge"

print("Repo root :", REPO_ROOT)
print("Knowledge :", KNOWLEDGE_DIR)


In [ ]:
products = pd.read_csv(Path("../knowledge/products.csv"))
print("products.csv shape:", products.shape)
print("Columns          :", list(products.columns))
print()
print("Category breakdown:")
print(products["category"].value_counts())
print()
print("Three example products:")
print(products[["product_id", "product_name", "category", "price"]].head(3).to_string(index=False))


 ## 3.3 Pipeline overview



 From a customer click on a product card to six recommendation cards

 rendering, the flow is:



 ```

 +----------------------------+    +-----------------------------+

 | Customer clicks a product  |--->|  Flask route                |

 | card on / or /category/... |    |  GET /product/<id>          |

 +----------------------------+    +--------------+--------------+

                                                  |

                                                  v

                               +------------------+-------------------+

                               |  SearchIndex.similar(product_id, 6)  |

                               |  in app/nlp/search.py                |

                               +------------------+-------------------+

                                                  |

                   +------------------------------+------------------------------+

                   |                              |                              |

                   v                              v                              v

         +---------+----------+        +----------+---------+         +----------+---------+

         |  Find the target's |        |  Score every other |         |  Add +0.1 to       |

         |  row in the matrix |        |  product by cosine |         |  same-category     |

         |  by product_id     |        |  similarity to it  |         |  candidates        |

         +---------+----------+        +----------+---------+         +----------+---------+

                                                  |

                                                  v

                                 +----------------+----------------+

                                 |  Exclude the target itself,     |

                                 |  sort descending, take top 6    |

                                 +----------------+----------------+

                                                  |

                                                  v

                                 +----------------+----------------+

                                 |  Render six product cards in    |

                                 |  the "Similar items" section    |

                                 |  (app/templates/product.html)   |

                                 +---------------------------------+

 ```



 ### The TF-IDF index is built once



 When the Flask app starts (`create_app()` in `app/__init__.py`), the

 `SearchIndex` is constructed once over the full catalogue. That step

 fits the TF-IDF vectorizer and stores the sparse matrix. After

 startup, the matrix is read-only so every recommendation request just

 does a dot-product against that pre-built matrix.



 ### Per-request cost is one row of cosines, not a full pairwise matrix



 For a recommendation request, we don't compute the full

 `N × N` similarity matrix at runtime. We compute one row, the target

 product's vector against the matrix which is `1 × N`. This is the

 key efficiency point covered in section 7.

 ## 3.4 Similarity score formula



 ### Step 1 vectorise each product



 For every product `i`, build a sparse vector `v_i` of TF-IDF weights

 over character n-grams of `product_name + " " + category`. We use:



 - `analyzer="char_wb"` — character n-grams restricted to within word

   boundaries (no n-grams that cross spaces).

 - `ngram_range=(3, 5)` — trigrams through 5-grams. Trigrams catch

   spelling overlap; 4- and 5-grams discriminate between similar-

   looking brands.

 - `lowercase=True`, `min_df=1` — keep even rare features; the

   catalogue is only 1,000 rows so dropping rare n-grams loses signal.



 This is the **same** vectorizer Task 1 (search) uses. Reusing it is

 deliberate — Task 1 and Task 3 ask the same underlying question

 ("how textually close are two product strings?") and a shared matrix

 keeps the answer consistent.



 ### Step 2 cosine similarity



 For a target product `t` and any candidate product `c`, the raw

 similarity is the cosine of the angle between their vectors:



 $$

 \text{cos\_sim}(t, c) \;=\; \frac{v_t \cdot v_c}{\lVert v_t \rVert \, \lVert v_c \rVert}

 $$



 In plain words: how much of the target's TF-IDF weight lines up with

 the candidate's, ignoring the absolute size of either vector. Cosine

 is scale-invariant, so a long product name doesn't outscore a short

 one just for being longer.



 The value is in `[0, 1]` (TF-IDF weights are non-negative, so there

 are no negative cosines).



 ### Step 3 category boost



 A small **+0.1** bonus is added when the candidate is in the same

 category as the target:



 $$

 \text{score}(t, c) \;=\; \text{cos\_sim}(t, c) \;+\; 0.1 \cdot \mathbf{1}\!\left[\text{category}(c) = \text{category}(t)\right]

 $$



 where `1[·]` is the indicator function (1 when the bracketed

 condition holds, 0 otherwise).



 In plain words: if the candidate is the same kind of product

 (Skincare ↔ Skincare, Makeup ↔ Makeup, ...), give it a small bump

 worth a tenth of a unit of cosine similarity.



 ### Step 4 exclude self, rank, take top-6



 The target's own row is set to a sentinel `-1` so the product can

 never recommend itself, then the score row is argsort-descended and

 the top 6 candidates are returned.



 ### Why +0.1 specifically



 - Small enough that a candidate with high textual similarity from a

   different category can still outrank a same-category candidate

   with weak similarity. We don't want to lock recommendations

   inside the category.

 - Large enough that **ties or near-ties** (cos ≈ cos within a few

   hundredths) tip in favour of the same-category candidate. This

   matters because char-n-gram lookalikes from unrelated categories

   do show up at the top of raw cosine rankings, and the customer

   intuition for "similar" leans towards "same kind of product".

 - A hand-picked constant. A production system would tune it from

   click-through-rate data.



 ### Why cosine, not Euclidean or dot product



 - **Euclidean distance** would penalise vectors with different

   magnitudes. Product names of different lengths produce vectors of

   different L2 norms; cosine normalises that away.

 - **Plain dot product** is even more affected by magnitude — long

   product names with many n-grams would dominate. Cosine

   normalises both vectors to unit length before taking the dot

   product.

 - **Cosine** is the textbook choice for sparse TF-IDF vectors. It is

   exactly the inner product after L2-normalisation.

 ## 3.5  Recommender code, with inline comments



 The code below is the **inlined, annotated** version of the function

 that ships to the live app in `app/nlp/search.py:SearchIndex.similar`.

 Every line that does work has a comment to its right or above

 explaining what it does and why.



 The cell defines a class with the same constructor + method signature

 the live app uses, so the marker can confirm the live behaviour from

 this notebook alone.

In [ ]:
class Recommender:
    """Inlined copy of app/nlp/search.py:SearchIndex (similar() side).

    Public API:
      - similar(product_id, top_n=6) -> list[dict]: the top-N most
        similar products to ``product_id``, excluding the target.

    Behaviour and parameters are identical to the live app's
    SearchIndex.similar() so the numbers reported in this notebook
    match what a marker would see on the live product-detail page.
    """

    # The category-boost constant. Documented in §4 of this notebook.
    CATEGORY_BOOST = 0.1

    def __init__(self, products: pd.DataFrame) -> None:
        # reset_index so iloc-based lookups line up with the matrix
        # row order. We never use the original DataFrame index after
        # this point.
        self._products = products.reset_index(drop=True)

        # The document for each product is "<name> <category>".
        # Joining the two short strings gives the vectorizer a
        # slightly richer signal than the name alone.
        documents = [
            f"{row.product_name} {row.category}"
            for row in self._products.itertuples(index=False)
        ]

        # char_wb (character n-grams restricted to within word
        # boundaries) + (3,5) ngram range = the same configuration
        # Task 1 uses. Sharing the vectorizer keeps search and
        # recommendations consistent.
        self._vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            lowercase=True,
            min_df=1,  # keep rare n-grams — the catalogue is small.
        )
        # fit_transform builds the sparse TF-IDF matrix. After this
        # line, self._matrix has shape (n_products, n_ngrams) and is
        # what every recommendation query dot-products against.
        self._matrix = self._vectorizer.fit_transform(documents)

    def similar(self, product_id: int, top_n: int = 6) -> list[dict]:
        """Return up to top_n products most similar to product_id."""

        # Find the target's position in the matrix. If product_id is
        # unknown, raise — the caller (the Flask route) has already
        # validated the id with a 404 before reaching here.
        mask = self._products["product_id"] == product_id
        if not mask.any():
            raise KeyError(f"product_id {product_id} not in index")
        target_pos = int(mask.idxmax())
        target_category = self._products.iloc[target_pos]["category"]

        # One row of the pairwise similarity matrix — the target's
        # vector against every product's vector. Shape (1, N) which
        # we flatten to (N,).
        scores = cosine_similarity(
            self._matrix[target_pos], self._matrix
        ).flatten()

        # +0.1 boost for same-category candidates. This is the
        # vectorised version of the indicator function in §4's
        # formula.
        same_cat = (self._products["category"] == target_category).to_numpy()
        scores = scores + self.CATEGORY_BOOST * same_cat.astype(float)

        # Exclude the target itself from its own recommendation list
        # by setting its score to a sentinel below any real score.
        scores[target_pos] = -1.0

        # argsort gives ascending order; reverse for descending and
        # take the first top_n entries. We could use np.argpartition
        # for asymptotic savings, but on N=1,000 the constant factor
        # of np.argsort wins.
        ranked = scores.argsort()[::-1]
        out: list[dict] = []
        for i in ranked:
            if len(out) >= top_n:
                break
            row = self._products.iloc[int(i)].to_dict()
            # Attach the score so the marker (or a future ranking
            # UI) can sort or threshold downstream.
            row["score"] = float(scores[i])
            out.append(row)
        return out


# Instantiate once. The constructor is the only step that runs the
# TF-IDF fit; every call to .similar(...) afterwards is a cheap dot
# product against the pre-built matrix.
recommender = Recommender(products)
print("Index built. Matrix shape:", recommender._matrix.shape)
print("  rows = products, cols = char n-grams in the vocabulary")


 ## 3.6  Worked example : actual output, before vs after the boost



 We pick a probe product, then show the top-6 most similar items

 **without** the category boost and **with** it side by side, in

 two tables.



 Reading the two tables, the marker can verify:



 - The boost preserves strong textual matches (a same-brand product

   that is already at the top stays at the top).

 - The boost re-orders ties and near-ties in favour of items in the

   same category as the probe.

 - The numerical score difference between the two tables is exactly

   `0.1` for same-category rows (the boost) and `0.0` for

   different-category rows.

In [ ]:
PROBE_ID = 120

probe_idx = products.index[products["product_id"] == PROBE_ID][0]
probe = products.iloc[probe_idx]
print(f"Probe product #{PROBE_ID}: {probe['product_name']}")
print(f"  Category: {probe['category']}")
print(f"  Price   : ${probe['price']:.2f}")
print()

# --- Compute the raw (no-boost) top-6 -----------------------------
raw_scores = cosine_similarity(
    recommender._matrix[probe_idx], recommender._matrix
).flatten()
raw_scores_excl = raw_scores.copy()
raw_scores_excl[probe_idx] = -1.0
raw_top6 = np.argsort(-raw_scores_excl)[:6]

raw_table = pd.DataFrame({
    "rank": np.arange(1, 7),
    "product_id":   products.iloc[raw_top6]["product_id"].values,
    "product_name": products.iloc[raw_top6]["product_name"].values,
    "category":     products.iloc[raw_top6]["category"].values,
    "raw_cosine":   raw_scores[raw_top6].round(3),
})

print("Top-6 BEFORE category boost (raw cosine only):")
print(raw_table.to_string(index=False))
print()

# --- The .similar() call returns the boosted ranking --------------
boosted = recommender.similar(PROBE_ID, top_n=6)
boosted_table = pd.DataFrame({
    "rank": np.arange(1, 7),
    "product_id":   [r["product_id"]   for r in boosted],
    "product_name": [r["product_name"] for r in boosted],
    "category":     [r["category"]     for r in boosted],
    "boosted_score":[round(r["score"], 3) for r in boosted],
})
print("Top-6 AFTER  category boost (cosine + 0.1 if same category):")
print(boosted_table.to_string(index=False))


 The bar chart below visualises the boosted scores for the six

 recommended cards in the order they appear on the website.

In [ ]:
labels = [n[:24] for n in boosted_table["product_name"]]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(labels[::-1], boosted_table["boosted_score"].values[::-1])
ax.set_xlabel("boosted similarity score")
ax.set_title(f"Top-6 similar items to product #{PROBE_ID} ({probe['product_name']})")
fig.tight_layout()
plt.show()


 ### Analysis



 - Before the boost, the top 6 may include cross-category items that

   share name n-grams (different products from the same brand).

 - After the boost, items in the same category as product 101 are

   preferred when there's a tie or near-tie in raw similarity. This

   matches the customer intuition that "similar" means "same kind of

   product" first, "same brand" second.



 ## 3.7 Efficiency and scalability

 ### One-time cost: building the index



 `SearchIndex.__init__` does:



 - `TfidfVectorizer.fit_transform` on N short documents → builds the

   char-n-gram vocabulary and produces the sparse matrix.

 - Cost: a fraction of a second for N=1,000 with `ngram_range=(3,5)`.



 This happens once at app startup, not per request.



 ### Memory footprint



 - The TF-IDF matrix is **sparse**. For N=1,000 products with short

   names, the non-zero count is ≈ N × (avg n-grams per product) ≈

   tens of thousands of entries. Single-digit MB in memory.

 - The vocabulary itself is small enough to keep in RAM with no

   paging concern.



 ### Why we don't precompute the full N × N matrix



 Two reasons:



 1. **Memory.** N × N dense floats grows quadratically. At N=1,000

    it is already 8 MB just for the floats; at N=100,000 it would be

    80 GB. Computing rows on demand stays linear in catalogue size.

 2. **Freshness.** If a product were added or its name edited, only

    the index needs to be rebuilt — not a stored full matrix that

    would now be stale at every cell touching the changed row.



 ### How this scales to a larger catalogue



 | N (catalogue size) | Per-request `.similar(...)` | Notes |

 |---|---|---|

 | 1,000 (today) | sub-millisecond | Template renders eclipse this. |

 | 10,000 | ~ few ms | Still well under a 50 ms per-page budget. |

 | 100,000 | ~ tens of ms | Consider an approximate-nearest-neighbours index (Annoy, FAISS) at this point. |

 | 1,000,000 | ~ hundreds of ms naively | Definitely move to ANN; possibly cluster pre-filter by category to drop the search universe. |



 So the current method is efficient for catalogues from "small demo"

 through "mid-sized boutique e-commerce" without modification, and

 the upgrade path (ANN, category pre-filter) is well-trodden.

 ## 3.8  App integration where Task 3 appears on the website



 ### Route



 - **URL pattern:** `GET /product/<int:product_id>`

 - **View function:** `product_detail()` in `app/__init__.py` (the

   Flask application factory).

 - **Trigger:** the customer clicks any product card — either on the

   home page (`/`), a category page (`/category/<slug>`), the search

   results page (`/search?q=…`), or even one of the recommendation

   cards on another product page.



 The view function passes the recommender's output into the template

 under the variable name `similar`:



 ```python

 @app.route("/product/<int:product_id>")

 def product_detail(product_id: int):

     ...

     # Task 3: similar-item recommendations (top 6, cosine + +0.1

     # same-category boost).

     similar = search_index.similar(product_id, top_n=6)

     ...

     return render_template(

         "product.html",

         product=product, reviews=review_list,

         similar=similar, aspects=aspects,

     )

 ```



 ### Template : where on the page



 The product-detail template (`app/templates/product.html`) renders

 the recommendations in a dedicated section **below the product hero

 and above the reviews list**. The relevant block:



 ```html

 {% if similar %}

 <section class="similar">

   <div class="section-title">

     <h2>Similar items</h2>

     <span class="section-title__meta">Ranked by similarity</span>

   </div>

   <div class="similar-grid">

     {% for s in similar %}

       {{ similar_card(s) }}

     {% endfor %}

   </div>

 </section>

 {% endif %}

 ```

 So on every product-detail page the customer sees, in order from top

 to bottom:



 1. **Breadcrumb** — "← Back to catalogue".

 2. **Product hero** — image, category, name, price, "Customers

    mention" aspect pills (Task 4), "Write a review" CTA.

 3. **Similar items** ← **Task 3 lands here**, six cards in a grid

    with the heading "Similar items · Ranked by similarity".

 4. **Reviews list** — the seed reviews + any reviews the customer

    has posted, with the Task 2 classifier label.



  ### Where to confirm in the codebase



 | Concern | File | Function / section |

 |---|---|---|

 | Index built at startup | `app/__init__.py` | `create_app()` → `SearchIndex(products)` |

 | `.similar(...)` definition | `app/nlp/search.py` | `SearchIndex.similar` |

 | Route that consumes it | `app/__init__.py` | `product_detail()` |

 | Template rendering | `app/templates/product.html` | `<section class="similar">` |

 | Card layout macro | `app/templates/_macros.html` | `similar_card(p)` |

 ## 3.9 Conclusion



 ### What Task 3 delivers



 - A **similar-item recommender** that produces six related products

   for every catalogue item, with no cold-start gaps (every product

   gets recommendations, even those with zero reviews).

 - A **defensible similarity measure** cosine over char-n-gram

   TF-IDF paired with a small category boost that nudges the

   ranking towards same-kind-of-product matches.

 - **Sub-millisecond per-page cost**, which leaves the per-request

   budget free for template rendering and other tasks.

 - A **clear customer-facing display**: six cards in a grid with the

   heading "Similar items · Ranked by similarity", each card showing

   the image, category, product name, and price, with the whole card

   clickable.



 ## 4 · Task 4 — Aspect Extraction ("Customers mention")



 ### Problem framing



 Milestone 2 Task 4 is the open-ended "additional functionality" slot.

 We surface, on each product-detail page, the words customers most

 distinctively use about *that* product — rendered as small

 "Customers mention …" pills. Customers can scan the pills in a

 fraction of the time it would take to read the reviews.



 ### Approach justification



 We use **per-product TF-IDF**. The corpus has one document per

 product (the aggregated review text of that product), and we fit a

 TF-IDF vectorizer across that corpus. The top-K words per row are

 the distinctive terms for that product.



 The two-part TF-IDF weighting does the work:



 - **TF (term frequency)** rewards words customers use a lot about

   this product.

 - **IDF (inverse document frequency)** discounts words that almost

   every product attracts — universal compliments like "good",

   "great", "love", "nice".



 What remains after both terms apply is what makes a product

 distinct.



 ### Why we don't do sentiment analysis here



 The pills surface *what customers talk about*, not whether the

 sentiment is positive. A future upgrade would do aspect-based

 sentiment (distinguish "customers mention dryness positively" from

 "customers mention dryness negatively"). Task 4 is a one-marker; we

 stayed within scope.



 ### Alternatives rejected



 - **Topic modelling (LDA, NMF).** Outputs topics, not per-product

   terms. Slower, and the topic-to-product assignment is fuzzy.

 - **LLM-based aspect mining.** Strongest semantic results but adds a

   model dependency and a per-page inference cost. Out of scope.

 - **Hand-picked aspect dictionaries.** Brittle, labour-intensive,

   and wouldn't generalise to new products.

In [ ]:
# Build the per-product corpus from the SEED reviews (the live app
# displays these reviews, so they're the right input to extract
# aspect terms from for the product-detail page).
per_product = (
    reviews_seed.dropna(subset=["review_text"])
                .groupby("product_id")["review_text"]
                .apply(lambda s: " ".join(s.astype(str)))
                .reset_index()
)
print(f"Products with ≥1 review: {len(per_product)} of {len(products)}")
print(per_product.head(3).to_string(index=False))


 ### Vectoriser configuration



 - `stop_words="english"` — drops common function words.

 - `min_df=2` — a term must appear in at least two products to be

   considered. This filter discards review-specific quirks and

   one-off proper nouns that aren't useful aspects.

 - `lowercase=True` — keeps the vocabulary compact.



 `min_df` could be tightened to 3 or higher for production. We

 picked 2 to keep the per-product top-K richer for the catalogue's

 relatively short review corpus.

In [ ]:
aspect_vec = TfidfVectorizer(stop_words="english", min_df=2, lowercase=True)
A = aspect_vec.fit_transform(per_product["review_text"])
vocab_aspect = np.array(aspect_vec.get_feature_names_out())
print("Aspect TF-IDF matrix:", A.shape, "(rows=products with reviews, cols=words)")


 ### Worked example — the most-reviewed product



 Pick the product with the largest review count and show its top-5

 aspect terms. The bar plot below visualises the TF-IDF score of each

 term so the marker can see the relative weights.

In [ ]:
review_counts = reviews_seed.groupby("product_id").size().sort_values(ascending=False)
probe_pid = int(review_counts.index[0])
probe_idx_in_A = per_product.index[per_product["product_id"] == probe_pid][0]
probe_product = products[products["product_id"] == probe_pid].iloc[0]

print(f"Probe product #{probe_pid}: {probe_product['product_name']}")
print(f"  Reviews aggregated: {review_counts[probe_pid]}")
print()

probe_vec = A[probe_idx_in_A].toarray().ravel()
top_n = 5
top_idx = np.argsort(-probe_vec)[:top_n]
top_terms = pd.DataFrame({
    "term": vocab_aspect[top_idx],
    "tfidf": probe_vec[top_idx].round(3),
})
print(top_terms.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(top_terms["term"][::-1], top_terms["tfidf"][::-1])
ax.set_xlabel("TF-IDF score")
ax.set_title(f"Top-{top_n} aspect terms — product #{probe_pid}")
fig.tight_layout()
plt.show()


 ### Analysis



 Read the surfaced terms. Genuine aspects ("hydrating", "lightweight",

 "matte") are useful pills; words that slipped through ("product",

 "use") are signs the stoplist is too small. The bar plot makes the

 TF-IDF score gap visible — the top term is typically 2-3× more

 weighted than the fifth.



 ### Limitations



 - **sklearn's English stoplist is small** — domain words like

   "product", "use", "really" leak through.

 - **No phrase detection.** "Long lasting" appears as two separate

   unigrams. Bigrams (`ngram_range=(1,2)`) would capture the phrase

   at the cost of vocabulary growth.

 - **No sentiment polarity.** Pills show "customers talk about X",

   not "customers like X". Aspect-based sentiment is a natural

   next step.

 - **Cold-start.** Products with zero reviews show no pills.



 ### App pointer



 `app/nlp/aspect_extractor.py:AspectExtractor.top_terms(product_id,

 top_n=5)` ships this logic. The product-detail template calls it

 during render to populate the "Customers mention" pills.



 ### Milestone 1 pointer



 Not directly applicable — Task 4 is a Milestone 2-original task.

 ## 5 · Conclusions



 ### Unifying thread



 All four Milestone 2 tasks lean on the **TF-IDF / BoW** family of

 representations:



 - **Task 1 (search)** — char-n-gram TF-IDF over `name + category`

   and over the product `description`, combined with a weighted

   cosine (0.7 name + 0.3 description) so brand queries and

   description queries both work.

 - **Task 2 (classifier)** — BoW count vectors using the Milestone 1

   vocabulary (text head + title head), plus a 2-column numeric head.

 - **Task 3 (recommendations)** — reuses Task 1's TF-IDF matrix +

   category boost.

 - **Task 4 (aspects)** — per-product word TF-IDF over aggregated

   reviews.



 This consistency is deliberate. One well-understood feature backbone,

 applied across surfaces, makes the system easier to reason about and

 keeps the deployable artifact small. The Milestone 1 work

 (`knowledge/milestone1_*.ipynb`) established that BoW + LR is a

 competitive baseline for the classification task; we reuse that

 foundation for Milestone 2's classifier and extend the same family

 of representations to the three other surfaces.



 ### Relationship to Milestone 1 — summary



 | M2 component | M1 contribution |

 |---|---|

 | Tokenizer (`app/nlp/preprocessing.py:tokenize`) | `knowledge/milestone1_task1.ipynb` §1.2 |

 | Vocabulary filters (hapax + top-20) | `knowledge/milestone1_task1.ipynb` §1.3 |

 | BoW + LR baseline | `knowledge/milestone1_task2_3.ipynb` §3.4 |

 | FastText + LightGBM (Heads A & B) | `knowledge/milestone1_task2_3.ipynb` §3.5 |



 ### Where the work could go next



 - **Sentence-transformer embeddings for Tasks 1 and 3.** Would lift

   recall on paraphrase ("lipstick" ≡ "lip color") that char n-grams

   miss. Cost: a multi-MB model and an embedding step per query.

 - **Learned field weights for Task 1.** The 0.7 / 0.3 name/description

   split is hand-picked; click-through-rate logs would let us tune it.

 - **Aspect-based sentiment for Task 4.** Distinguish "customers

   mention X positively" from "customers mention X negatively". The

   current pills show only mention frequency.

 - **Expanded metadata for Task 2.** Incorporating additional Nykaa

   columns (`product_rating_count`, `brand`, …) into Head C would

   require those fields to exist in `products.csv` so the webapp can

   supply them at predict time.



 ### Running the live app



 The README walks through setup, training, image generation, and

 launching the Flask dev server (`python run.py`). All four NLP

 techniques above are wired into the routes documented in the

 README's "Routes / URLs exposed" table.